In [0]:
!pip install -U pypdf langchain-text-splitters pandas databricks-langchain

INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 816.1/816.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 

In [0]:
dbutils.library.restartPython() 

In [0]:
import os

print(os.getcwd())

/Workspace/Users/rahuljha.stats@gmail.com/rag_demo_databricks


In [0]:
from pypdf import PdfReader

In [0]:
import pypdf

In [0]:
import os

base_path = os.path.join(os.getcwd(), "pdfs")

pages = []

for file_name in os.listdir(base_path):
    full_path = os.path.join(base_path, file_name)
    reader = PdfReader(full_path)
    for page_num, page in enumerate(reader.pages, start = 1):
        text = page.extract_text()
        pages.append({
            "text" : text,
            "page_num" : page_num
        })

print(len(pages))



952


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    separators=["\n\n", "\n", " "]
)

In [0]:
chunks = []

for i, page in enumerate(pages):
    text = page["text"]
    chunks_subset = splitter.split_text(text)
    for j, chunk in enumerate(chunks_subset):
        chunks.append({
            "chunk" : chunk,
            "id" : f'chunk_id_{page["page_num"]}_{j}'
        })

print(len(chunks))

5070


In [0]:
chunks[100]

{'chunk': 'Queries                                                                                                                          272\nWhat Is a Query?                                                                                                      273\nThe Life of a Query                                                                                                  274\nThe Query Optimizer                                                                                              275',
 'id': 'chunk_id_15_6'}

In [0]:
import pandas as pd

data = pd.DataFrame(chunks)
data.head()

,chunk,id
0,Joe Reis & \n Matt Housley\nFundamentals \nof ...,chunk_id_1_0
1,DATA\n“The world of data has been \nevolving f...,chunk_id_2_0
2,"introduction to the business \nof moving, proc...",chunk_id_2_1
3,ISBN: 978-1-098-10830-4\nTwitter: @oreillymedi...,chunk_id_2_2
4,customers by evaluating the best technologies ...,chunk_id_2_3


In [0]:
from databricks_langchain import DatabricksEmbeddings

model = DatabricksEmbeddings(endpoint = "databricks-bge-large-en")



In [0]:
import numpy as np

In [0]:
np.array(model.embed_query("what is the meaning of Life?")).shape

(1024,)

In [0]:
np.array(model.embed_query(["what is the meaning of Life?"])).shape

---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-31690239-2e25-4c8e-81a5-8b3f28e4257f/lib/python3.12/site-packages/mlflow/utils/request_utils.py:108, in augmented_raise_for_status(response)
    107 try:
--> 108     response.raise_for_status()
    109 except HTTPError as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-31690239-2e25-4c8e-81a5-8b3f28e4257f/lib/python3.12/site-packages/requests/models.py:1028, in Response.raise_for_status(self)
   1027 if http_error_msg:
-> 1028     raise HTTPError(http_error_msg, response=self)

HTTPError: 400 Client Error: BAD_REQUEST: Parameter 'input' must be a string or a list of strings for embeddings models. for url: https://dbc-45c832dd-1442.cloud.databricks.com/serving-endpoints/databricks-bge-large-en/invocations

During handling of the above exception, another exception occurred:

HTTPError   

In [0]:
np.array(model.embed_documents(["what is the meaning of Life?", "what is the meaning of Life?", "what is the meaning of Life?", "what is the meaning of Life?"])).shape

(4, 1024)

In [0]:
texts = data["chunk"].tolist()

In [0]:
embds = np.array(model.embed_documents(texts))

In [0]:
embds.shape

(5070, 1024)

In [0]:
def normalize(vector):
    norms = np.linalg.norm(vector, axis = 1, keepdims = True)
    norms[norms == 0] = 1e-12
    return vector / norms

In [0]:
chunk_vectors = normalize(embds)

In [0]:
chunk_vectors.shape

(5070, 1024)

In [0]:
def retrieve(query, k = 3):
    query_vector = np.array(model.embed_query(query)).reshape((1,-1))
    normalized_query = normalize(query_vector)
    scores = (chunk_vectors @ normalized_query.T).flatten()

    top_ids = np.argsort(scores)[::-1][:k]

    top_matches_results = []
    for i in top_ids:
        top_matches_results.append(data.iloc[i].to_dict())
    return top_matches_results

In [0]:
retrieve("what is java?", k = 3)

[{'chunk': 'It’s known as “the second-best language at everything. ” Python underlies popular\ndata tools such as pandas, NumPy, Airflow, sci-kit learn, TensorFlow, PyTorch,\nand PySpark. Python is the glue between underlying components and is fre‐\nquently a first-class API language for interfacing with a framework.\nJVM languages such as Java and Scala\nPrevalent for Apache open source projects such as Spark, Hive, and Druid.\nThe JVM is generally more performant than Python and may provide access to',
  'id': 'chunk_id_46_2'},
 {'chunk': 'The JVM is generally more performant than Python and may provide access to\nlower-level features than a Python API (for example, this is the case for Apache\nSpark and Beam). Understanding Java or Scala will be beneficial if you’re using a\npopular open source data framework.\nbash\nThe command-line interface for Linux operating systems. Knowing bash com‐\nmands and being comfortable using CLIs will significantly improve your pro‐',
  'id': 'chunk_

In [0]:
retrieve("what is linux??", k = 3)

[{'chunk': 'xxviii   Introduction  \ndo and how they got that way. Linux is not just a piece of software; it’s also \na small part of the larger Unix culture, which has its own language and his -\ntory. I might throw in a rant or two, as well.\nThis book is divided into four parts, each covering some aspect of the \ncommand line experience.\n•\t Part 1, “Learning the Shell,” starts our exploration of the basic lan -\nguage of the command line including such things as the structure of',
  'id': 'chunk_id_30_0'},
 {'chunk': '1\nWhat Is the shell?\nWhen we speak of the command line, we \nare really referring to the shell. The shell is \na program that takes keyboard commands \nand passes them to the operating system to \ncarry out. Almost all Linux distributions supply a shell \nprogram from the GNU Project called bash. The name \nis an acronym for bourne-again shell, a reference to the \nfact that bash is an enhanced replacement for sh, the \noriginal Unix shell program written by Steve 

In [0]:
normalized_query.shape

(1, 1024)

In [0]:
scores

array([0.44410803, 0.45692314, 0.42666039, ..., 0.51926841, 0.4194044 ,
       0.48639406])

array([ 274,  275, 2569])

[{'chunk': 'It’s known as “the second-best language at everything. ” Python underlies popular\ndata tools such as pandas, NumPy, Airflow, sci-kit learn, TensorFlow, PyTorch,\nand PySpark. Python is the glue between underlying components and is fre‐\nquently a first-class API language for interfacing with a framework.\nJVM languages such as Java and Scala\nPrevalent for Apache open source projects such as Spark, Hive, and Druid.\nThe JVM is generally more performant than Python and may provide access to',
  'id': 'chunk_id_46_2'},
 {'chunk': 'The JVM is generally more performant than Python and may provide access to\nlower-level features than a Python API (for example, this is the case for Apache\nSpark and Beam). Understanding Java or Scala will be beneficial if you’re using a\npopular open source data framework.\nbash\nThe command-line interface for Linux operating systems. Knowing bash com‐\nmands and being comfortable using CLIs will significantly improve your pro‐',
  'id': 'chunk_

In [0]:
(chunk_vectors @ normalized_query.T).shape

(5070, 1)

In [0]:
query_vector.shape

(1024, 1)

In [0]:
normalized_query

In [0]:
query_vector.shape

(1024,)

In [0]:
normalized_embds

array([[-0.00226914,  0.01423268,  0.01248601, ..., -0.01894639,
         0.03529948,  0.02304992],
       [ 0.01871322,  0.00400671, -0.00135083, ..., -0.03901386,
        -0.00520872,  0.00828816],
       [-0.01740968,  0.00745366,  0.00173658, ..., -0.00697303,
         0.0190118 ,  0.01664677],
       ...,
       [ 0.00567486, -0.01905105,  0.01801301, ..., -0.01914264,
         0.0094721 , -0.01533396],
       [ 0.03708509,  0.00779092, -0.02975964, ..., -0.02307517,
        -0.02005342,  0.00558566],
       [ 0.03051178, -0.01219708,  0.02651474, ..., -0.01433291,
         0.02474505,  0.00922219]])

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint = "databricks-meta-llama-3-3-70b-instruct",
    temperature = 0.1,
    max_token = 500
)

In [0]:
def create_prompt(retrieved_sources, question):
    context = ""
    for i, r_s in enumerate(retrieved_sources):
        context += f"SOURCE.{i+1} {r_s['chunk']}"
    prompt = f"You are a helful assityant who asnwer the query in a professional manner only grounded in probided context. This is the question: {question} and this is the source: {context}. Do not move away from the topic"

    return prompt

In [0]:
def RAG(query):
    retrieved_sources = retrieve(query, k = 4)
    prompt = create_prompt(retrieved_sources, query)
    response = chat_model.invoke(prompt)
    answer = response.content

    return {
        "question" : query,
        "retrieved_sources" : retrieved_sources,
        "answer" : answer
    }

In [0]:
RAG("what is the capital of France")

{'question': 'what is the capital of France',
 'retrieved_sources': [{'chunk': '12\nA Gentle Introduct I on to v I\nThere is an old joke about a visitor to \nNew York City asking a passerby for direc-\ntions to the city’s famous classical music \nvenue.\nVisitor: Excuse me, how do I get to Carnegie Hall? \nPasserby: Practice, practice, practice!\nLearning the Linux command line, like becoming an accomplished \npianist, is not something that we pick up in an afternoon. It takes years of \npractice. In this chapter, we will introduce the vi (pronounced “vee eye”)',
   'id': 'chunk_id_159_0'},
  {'chunk': 'Ubuntu    6.06        06/01/2006\nSUSE      10.1        05/11/2006\nFedora    5           03/20/2006\nBy specifying -k 3.7, we instruct sort to use a sort key that begins at the \nseventh character within the third field, which corresponds to the start of \nthe year. Likewise, we specify -k 3.1 and -k 3.4 to isolate the month and day \nportions of the date. We also add the n and r optio

In [0]:
RAG("what is AI?")

{'question': 'what is AI?',
 'retrieved_sources': [{'chunk': 'The world of ML engineering is snowballing and parallels a lot of the same develop‐\nments occurring in data engineering. Whereas several years ago, the attention of ML\nwas focused on how to build models, ML engineering now increasingly emphasizes\nincorporating best practices of machine learning operations (MLOps) and other\nmature practices previously adopted in software engineering and DevOps.\nAI researchers work on new, advanced ML techniques. AI researchers may',
   'id': 'chunk_id_54_0'},
  {'chunk': 'AI researchers work on new, advanced ML techniques. AI researchers may\nwork inside large technology companies, specialized intellectual property startups\n(OpenAI, DeepMind), or academic institutions. Some practitioners are dedicated\nto part-time research in conjunction with ML engineering responsibilities inside a\ncompany. Those working inside specialized ML labs are often 100% dedicated to\nresearch. Research probl